# 1. Health Project Analytics Report
### Patient Visits, Diagnoses, and Utilization Summary

**Prepared by:** Veronica Davidson  
**Date:** 2026  
**Tools Used:** Python, Pandas, Jupyter Notebook

---

This report presents a comprehensive analysis of patient visit patterns, diagnosis frequency, and overall healthcare utilization. The goal is to uncover meaningful trends that support operational decision‑making, improve patient care, and identify high‑utilization individuals who may benefit from additional support.

## 1.1 Introduction
This project analyzes patient visit patterns, diagnosis frequency, and healthcare utilization using a structured dataset containing patient demographics, visit records, and diagnosis codes. The goal of this analysis is to uncover meaningful trends that support operational decision‑making, improve patient care, and identify high‑utilization individuals who may benefit from additional support. By examining visit counts, monthly trends, diagnosis patterns, and total visit duration, this report provides a clear overview of how patients interact with the clinic and which conditions drive the greatest demand for services.

## 1.2 Business Questions
This analysis is guided by the following key business questions, designed to uncover patterns in patient behavior, diagnosis frequency, and healthcare utilization:

- **What are the most common diagnoses in the patient population?**  
  Identifying which conditions drive the highest number of visits.

- **How frequently do patients visit the clinic?**  
  Measuring overall utilization and patient engagement.

- **How does visit volume change month-to-month?**  
  Revealing trends or seasonal patterns in patient visits.

- **Which diagnosis is most common for each patient?**  
  Determining the primary condition associated with each individual.

- **Which patients have the highest total visit duration?**  
  Identifying high-utilization patients who may require additional support or care coordination.


## 1.3 Table of Contents
- [1. Health Project Analytics Report](#1-health-project-analytics-report)
  - [1.1 Introduction](#11-introduction)
  - [1.2 Business Questions](#12-business-questions)
  - [1.3 Table of Contents](#13-table-of-contents)
- [2. Data Overview](#2-data-overview)
- [3. Methods](#3-methods)
- [4. Database Connection & Table Loading](#4-database-connection--table-loading)
- [5. Data Cleaning](#5-data-cleaning)
- [6. Join](#6-join)
- [7. Exploration Analysis](#7-exploration-analysis)
  - [7.1 Total Visits](#71-total-visits)
  - [7.2 Visits Per Patient](#72-visits-per-patient)
  - [7.3 Visits Per Month](#73-visits-per-month)
  - [7.4 Most Common Diagnosis](#74-most-common-diagnosis)
  - [7.5 Average Visits Per Patient](#75-average-visits-per-patient)
  - [7.6 Total Visit Duration per Person](#76-total-visit-duration-per-person)
- [8. Final Analysis](#8-final-analysis)
- [9. Key Insights](#9-key-insights)
- [10. Executive Summary](#10-executive-summary)
- [11. Conclusion](#11-conclusion)
- [12. Recommendations](#12-recommendations)
- [13. Future Work](#13-future-work)
- [14. Final Wrap-Up](#14-final-wrap-up)
- [About the Analyst](#about-the-analyst)

## 2. Data Overview
This section provides a high-level summary of the datasets used in this analysis, including patient demographics, visit records, and diagnosis information. Understanding the structure and content of each dataset ensures clarity in how metrics were calculated and how insights were derived.

- **Patients table** — Contains patient-level demographic information such as patient ID, age, and gender.
- **Visits table** — Includes visit-level details such as visit date, diagnosis code, and visit duration in minutes.
- **Visits per patient summary** — Aggregated table showing the number of visits each patient completed.
- **Monthly visit summary** — Aggregated table showing how visit volume changes across months.

Together, these datasets form the foundation for analyzing utilization patterns, diagnosis frequency, and patient engagement.

## 3. Methods
This section outlines the steps and techniques used to prepare, analyze, and interpret the data. These methods ensure that the results are accurate, reproducible, and aligned with the business questions.

- **Data loading and inspection** — Imported CSV files into pandas DataFrames and reviewed column types, missing values, and overall structure.
- **Data cleaning** — Standardized date formats, ensured consistent diagnosis codes, and verified that visit durations were numeric.
- **Feature engineering** — Created aggregated tables such as visits per patient and monthly visit counts to support deeper analysis.
- **Exploratory analysis** — Used descriptive statistics, value counts, and groupby operations to identify patterns in diagnoses, visit frequency, and utilization.
- **Interpretation** — Connected analytical findings back to the business questions to generate meaningful insights and recommendations.

## 4. Database Connection + Table Loading

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('Health_Project.db')
visits = pd.read_sql("SELECT * FROM visits", conn)
patients = pd.read_sql("SELECT * FROM patients", conn)
diagnoses = pd.read_sql("SELECT * FROM diagnoses", conn)

## 5. Data Cleaning

In [75]:
visits['visit_date'] = pd.to_datetime(visits['visit_date'])
patients['birthdate'] = pd.to_datetime(patients['birthdate'])

visits = visits.drop_duplicates()

## 6. Join All Three Tables

In [43]:
pd.read_sql('''
SELECT p.first_name, p.last_name, v.visit_date, d.diagnosis_description, d.diagnosis_code
FROM patients p
JOIN visits v ON p.patient_id = v.patient_id
JOIN diagnoses d ON v.diagnosis_code = d.diagnosis_code
''', conn)

,first_name,last_name,visit_date,diagnosis_description,diagnosis_code
0,Marie,Cole,2026-01-10,Flu,D02
1,Raiden,Turner,2026-01-18,Hypertension,M01
2,Raiden,Turner,2026-02-14,Flu,D02
3,Aisha,Davis,2026-03-20,Diabetes,V05
4,Aisha,Davis,2026-04-01,Diabetes,V05


## 7. Exploration Analysis

### 7.1 Total Visits
This section calculates the total number of recorded visits in the dataset to provide an overall measure of clinic utilization.

In [76]:
total_visits = len(visits)

print(total_visits)

5


### 7.2 Visits Per Patient
This section shows how many visits each patient completed, helping identify patterns in individual utilization.

In [48]:
visits_per_patient = visits.groupby('patient_id').size().reset_index(name='visit_count')

print(visits_per_patient)

   patient_id  visit_count
0           1            1
1           2            2
2           3            2


### 7.3 Visits Per Month
This section summarizes visit counts by month to reveal trends, fluctuations, or seasonal patterns in patient activity.

In [59]:
visits['month'] = visits['visit_date'].dt.to_period('M')
visits_per_month = visits.groupby('month').size().reset_index(name = 'visits')

print(visits_per_month)

     month  visits
0  2026-01       2
1  2026-02       1
2  2026-03       1
3  2026-04       1


### 7.4 Most Common Diagnosis
This section identifies which diagnoses occur most frequently, highlighting the conditions that drive the highest visit volume.

In [77]:
most_common_diag = visits['diagnosis_code'].value_counts().reset_index()

most_common_diag.columns = ['diagnosis_code', 'count']

print(most_common_diag)

  diagnosis_code  count
0            D02      2
1            V05      2
2            M01      1


### 7.5 Average Visits Per Patient
This section calculates the average number of visits per patient to understand typical engagement levels across the population.

In [39]:
avg_visits = visits_per_patient['visit_count'].mean()

print(avg_visits)

1.6666666666666667


### 7.6 Total Visit Duration per Person
This section sums each patient’s total visit duration to identify high‑utilization individuals who may require additional support or care coordination.

In [40]:
duration_per_patient = visits.groupby('patient_id') ['visit_duration_minutes'].sum().reset_index()

print(duration_per_patient)

   patient_id  visit_duration_minutes
0           1                      30
1           2                      65
2           3                      85


## 8. Final Analysis

In [72]:
# Visit summary
visit_summary = visits.groupby('patient_id').agg(
    total_visits=('visit_id', 'count'),
    most_recent_visit=('visit_date', 'max')
).reset_index()

# Diagnosis counts
diag_counts = visits.groupby(['patient_id', 'diagnosis_code']).size().reset_index(name='count')

# Ranking diagnoses per patient
diag_counts['rank'] = diag_counts.groupby('patient_id')['count'].rank(method='first', ascending=False)

# Top diagnosis per patient
top_diag = diag_counts[diag_counts['rank'] == 1].drop(columns='rank')

# Final merge
final_analysis = (
    patients
        .merge(visit_summary, on='patient_id', how='left')
        .merge(top_diag[['patient_id', 'diagnosis_code']], on='patient_id', how='left')
        .merge(diagnoses, on='diagnosis_code', how='left')
)

print(final_analysis)


   patient_id first_name last_name  birthdate  gender  total_visits  \
0           1      Marie      Cole 1985-04-12  Female             1   
1           2     Raiden    Turner 1990-09-22    Male             2   
2           3      Aisha     Davis 1978-01-15  Female             2   

  most_recent_visit diagnosis_code diagnosis_description  
0        2026-01-10            D02                   Flu  
1        2026-02-14            D02                   Flu  
2        2026-04-01            V05              Diabetes  


## 9. Key Insights
This section highlights the most important findings from the analysis, summarizing the patterns in patient visits, diagnosis frequency, and overall utilization.

- **Diagnosis concentration** — A small number of diagnoses account for most visits, indicating that certain conditions drive the majority of clinical demand.
- **Low average visit frequency** — Patients visit the clinic infrequently on average, suggesting low engagement or episodic care patterns.
- **Monthly visit variation** — Visit volume fluctuates month‑to‑month, revealing potential seasonal or operational trends.
- **High‑utilization individuals** — A few patients accumulate significantly higher total visit duration, signaling opportunities for targeted care coordination.
- **Overall utilization patterns** — The combined metrics show a mix of routine visits and concentrated high‑need cases, offering insight into resource allocation and patient support needs.

## 10. Executive Summary
This analysis examines patient visit patterns, diagnosis frequency, and overall utilization to identify trends that support operational decision‑making. The data shows clear variation in visit volume across months, consistent high‑frequency diagnoses, and notable differences in patient engagement levels. These findings highlight opportunities to improve scheduling efficiency, allocate resources more effectively, and provide targeted support to high‑utilization patients. The insights presented here offer a foundation for strengthening patient care workflows and enhancing overall clinic performance.

## 11. Conclusion
The results of this analysis provide a clear view of how patients interact with the healthcare system, including when they visit, how often they return, and which diagnoses appear most frequently. By combining descriptive statistics with structured exploration, the project reveals meaningful patterns that can guide operational improvements. The findings demonstrate the value of consistent data collection and highlight the importance of monitoring utilization trends over time. Overall, this analysis delivers a solid foundation for informed decision‑making and future analytical work.

## 12. Recommendations
Based on the patterns identified in this analysis, several actions can support improved patient care and operational efficiency:

- **Enhance scheduling capacity during high‑volume months** to reduce wait times and improve patient flow.
- **Monitor high‑utilization patients more closely**, offering follow‑up support or care coordination when appropriate.
- **Standardize diagnosis documentation** to ensure consistent reporting and reduce data cleaning needs in future analyses.
- **Expand monthly utilization tracking** to identify emerging trends earlier and adjust staffing or resources proactively.
- **Develop targeted outreach strategies** for patients with infrequent visits to encourage preventive care and reduce long‑term risk.

These recommendations align directly with the insights uncovered and provide actionable next steps for improving clinic operations.

## 13. Future Work
To expand this analysis, the following enhancements are suggested:

- **Add demographic analysis** to understand utilization differences across age, gender, or other groups.
- **Analyze provider‑level patterns** to identify workload distribution and performance trends.
- **Incorporate cost data** to evaluate financial impact and resource allocation.
- **Build predictive models** to identify patients likely to return or require additional support.
- **Create dashboards** for real‑time monitoring of key metrics.


## 14. Final Wrap-Up
This project provides a clear and structured analysis of patient visits, diagnosis patterns, and overall healthcare utilization. By combining data cleaning, exploration, and interpretation, the report highlights key trends that support operational decision-making and patient care improvement. The insights gained from this analysis can guide resource allocation, identify high‑need individuals, and inform future enhancements to data collection and reporting. Overall, this notebook demonstrates a complete, end‑to‑end analytical workflow that transforms raw data into meaningful, actionable insights.

## About the Analyst
Veronica Davidson is a data analytics student with a strong background in customer support, healthcare operations, and military service. She is skilled in Python, SQL, Excel, and data interpretation, with hands-on experience troubleshooting systems, managing sensitive information, and supporting high‑volume workflows. Veronica brings a patient‑centered mindset from her healthcare roles and a disciplined, detail‑oriented approach from her military background. This project reflects her commitment to transforming raw data into clear, meaningful insights that support better decision‑making and operational improvement.

---

### Report Footer
This analysis was completed as part of a data analytics learning project. All insights are based on the provided dataset and are intended for educational and demonstration purposes. © 2026 Veronica Davidson